# Assignment 4 - Feature Selection and Dimensionality Reduction

This notebook demonstrates:
1. Sequential Backward Selection (SBS) with KNN for top 5 features
2. Top 5 important features based on Random Forest
3. Top 5 features based on Linear PCA

Each method's selected features are evaluated using a Linear SVM classifier.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

## 1. Load Dataset, Stratify Split, and Standardize

In [2]:
# Load the Breast Cancer Wisconsin dataset
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Class distribution: {np.bincount(y)}")

Dataset shape: (569, 30)
Number of features: 30
Number of samples: 569
Class distribution: [212 357]


In [3]:
# Stratify split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

Training set size: 455
Testing set size: 114


In [4]:
# Standardize features
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

print("Features standardized successfully.")

Features standardized successfully.


## 2. Sequential Backward Selection (SBS) with KNN - Top 5 Features

In [5]:
# Sequential Backward Selection using KNN classifier
knn = KNeighborsClassifier(n_neighbors=5)

sbs = SFS(knn,
          k_features=5,
          forward=False,
          floating=False,
          scoring='accuracy',
          cv=5)

sbs.fit(X_train_std, y_train)

# Get selected feature indices and names
sbs_feature_indices = list(sbs.k_feature_idx_)
sbs_feature_names = list(sbs.k_feature_names_)

print("SBS with KNN - Selected top 5 feature indices:", sbs_feature_indices)
print("SBS with KNN - Selected top 5 feature names:", [feature_names[i] for i in sbs_feature_indices])

SBS with KNN - Selected top 5 feature indices: [0, 6, 8, 15, 21]
SBS with KNN - Selected top 5 feature names: [np.str_('mean radius'), np.str_('mean concavity'), np.str_('mean symmetry'), np.str_('compactness error'), np.str_('worst texture')]


## 3. Top 5 Important Features Based on Random Forest

In [6]:
# Fit Random Forest and get feature importances
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_std, y_train)

# Get top 5 feature indices by importance
importances = rf.feature_importances_
rf_feature_indices = np.argsort(importances)[::-1][:5].tolist()

print("Random Forest - Top 5 feature indices:", rf_feature_indices)
print("Random Forest - Top 5 feature names:", [feature_names[i] for i in rf_feature_indices])
print("Random Forest - Top 5 feature importances:", [round(importances[i], 4) for i in rf_feature_indices])

Random Forest - Top 5 feature indices: [23, 27, 20, 7, 22]
Random Forest - Top 5 feature names: [np.str_('worst area'), np.str_('worst concave points'), np.str_('worst radius'), np.str_('mean concave points'), np.str_('worst perimeter')]
Random Forest - Top 5 feature importances: [np.float64(0.14), np.float64(0.1295), np.float64(0.0977), np.float64(0.0909), np.float64(0.0722)]


## 4. Top 5 Features Based on Linear PCA

In [7]:
# Apply Linear PCA to reduce to 5 components
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

print("PCA - Number of components: 5")
print("PCA - Explained variance ratio:", [round(v, 4) for v in pca.explained_variance_ratio_])
print("PCA - Total explained variance: {:.4f}".format(sum(pca.explained_variance_ratio_)))

PCA - Number of components: 5
PCA - Explained variance ratio: [np.float64(0.4441), np.float64(0.1894), np.float64(0.0954), np.float64(0.0672), np.float64(0.0552)]
PCA - Total explained variance: 0.8514


## 5. Linear SVM Accuracies with Selected Features

In [8]:
# Helper function to train Linear SVM and print accuracies
def evaluate_svm(X_train_selected, X_test_selected, y_train, y_test, method_name):
    svm = SVC(kernel='linear', random_state=42)
    svm.fit(X_train_selected, y_train)
    train_acc = svm.score(X_train_selected, y_train)
    test_acc = svm.score(X_test_selected, y_test)
    print(f"\n{'='*60}")
    print(f"Linear SVM with {method_name}")
    print(f"{'='*60}")
    print(f"  Training Accuracy: {train_acc:.4f}")
    print(f"  Testing Accuracy:  {test_acc:.4f}")
    return train_acc, test_acc

In [9]:
# 1. Linear SVM with SBS (KNN) selected features
X_train_sbs = X_train_std[:, sbs_feature_indices]
X_test_sbs = X_test_std[:, sbs_feature_indices]
sbs_train_acc, sbs_test_acc = evaluate_svm(
    X_train_sbs, X_test_sbs, y_train, y_test,
    "Sequential Backward Selection (KNN) - Top 5 Features"
)

# 2. Linear SVM with Random Forest selected features
X_train_rf = X_train_std[:, rf_feature_indices]
X_test_rf = X_test_std[:, rf_feature_indices]
rf_train_acc, rf_test_acc = evaluate_svm(
    X_train_rf, X_test_rf, y_train, y_test,
    "Random Forest - Top 5 Important Features"
)

# 3. Linear SVM with PCA features
pca_train_acc, pca_test_acc = evaluate_svm(
    X_train_pca, X_test_pca, y_train, y_test,
    "Linear PCA - Top 5 Components"
)


Linear SVM with Sequential Backward Selection (KNN) - Top 5 Features
  Training Accuracy: 0.9714
  Testing Accuracy:  0.9386

Linear SVM with Random Forest - Top 5 Important Features
  Training Accuracy: 0.9560
  Testing Accuracy:  0.9298

Linear SVM with Linear PCA - Top 5 Components
  Training Accuracy: 0.9802
  Testing Accuracy:  0.9386


In [10]:
# Summary table
print("\n" + "="*70)
print("SUMMARY: Linear SVM Accuracies with Different Feature Selection Methods")
print("="*70)
print(f"{'Method':<45} {'Train Acc':>10} {'Test Acc':>10}")
print("-"*70)
print(f"{'Sequential Backward Selection (KNN)':<45} {sbs_train_acc:>10.4f} {sbs_test_acc:>10.4f}")
print(f"{'Random Forest Feature Importance':<45} {rf_train_acc:>10.4f} {rf_test_acc:>10.4f}")
print(f"{'Linear PCA (5 components)':<45} {pca_train_acc:>10.4f} {pca_test_acc:>10.4f}")
print("="*70)


SUMMARY: Linear SVM Accuracies with Different Feature Selection Methods
Method                                         Train Acc   Test Acc
----------------------------------------------------------------------
Sequential Backward Selection (KNN)               0.9714     0.9386
Random Forest Feature Importance                  0.9560     0.9298
Linear PCA (5 components)                         0.9802     0.9386
